# Análise dos resultados — RoboCar

Notebook único de análise. Roda de cima a baixo.

- **Seção 0** — setup (clone do repositório e dependências, apenas no Colab)
- **Seção 1** — corridas com o robô: percepção, erros e latência do pipeline
- **Seção 2** — comparação de latência entre os 4 modelos no vídeo da RoboCar

A Seção 1 roda em qualquer máquina, em segundos. A Seção 2 carrega os
modelos e processa vídeo — no Colab, ative a GPU em
*Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU*.

## Seção 0 — Setup

No Colab, clona o repositório e instala o Ultralytics. Localmente, basta
rodar o notebook a partir da raiz do repositório: a célula não faz nada.

In [1]:
import os
from pathlib import Path

# No Colab o repositorio nao existe: clona. Localmente, so confere a raiz.
if not Path("data/experimentos").exists():
    REPO = "https://github.com/vitor-domeniquelli/robocar-sinalizacao.git"
    !git clone -q {REPO} /content/repo
    %pip install -q ultralytics
    os.chdir("/content/repo")

print("diretorio de trabalho:", os.getcwd())
print("data/experimentos existe:", Path("data/experimentos").exists())
print("models/ existe          :", Path("models").exists())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 7.0 MB/s eta 0:00:00
diretorio de trabalho: /content/repo
data/experimentos existe: True
models/ existe          : True


## Seção 1 — Corridas com o robô

60 corridas, 10 por classe. Cada corrida gravou `frames.csv` (uma linha por
quadro processado), `meta.json` (veredito da corrida) e `video.mp4`.

### 1.1 — Carregamento

Duas tabelas com granularidades diferentes: `corridas` tem uma linha por
run e vem dos `meta.json`; `frames` tem uma linha por quadro e vem dos
`frames.csv`, com a run marcada para poder cruzar as duas.

In [2]:
import json
from pathlib import Path
import pandas as pd

RAIZ = Path("data/experimentos")

corridas = []   # uma linha por run (meta.json)
frames = []     # uma linha por frame (frames.csv), com a run marcada

for pasta in sorted(RAIZ.glob("*/run*")):
    meta_p = pasta / "meta.json"
    csv_p = pasta / "frames.csv"
    if not meta_p.exists() or not csv_p.exists():
        print(f"incompleta, pulando: {pasta}")
        continue

    m = json.loads(meta_p.read_text())
    m["classe_pasta"] = pasta.parent.name   # redundante com meta, mas garante
    corridas.append(m)

    f = pd.read_csv(csv_p)
    f["run"] = pasta.name
    f["classe_pasta"] = pasta.parent.name
    frames.append(f)

corridas = pd.DataFrame(corridas)
frames = pd.concat(frames, ignore_index=True)

print(f"corridas: {len(corridas)}   frames: {len(frames)}")
print()
print(corridas["classe_pasta"].value_counts().to_string())

corridas: 60   frames: 1260

classe_pasta
R-25a_esquerda            10
R-25b_direita             10
R-26_frente               10
R-3_sentido-proibido      10
R-4a_proibido-esquerda    10
R-4b_proibido-direita     10


### 1.2 — Percepção

`decisao_correta` é verdadeiro quando o modelo identificou a placa e o
mapeamento classe → comando produziu o comando esperado para aquela classe.
É a métrica da cadeia percepção + decisão, antes da execução física.

In [3]:
percepcao = (corridas
             .groupby("classe_pasta")
             .agg(n=("run", "count"),
                  acertos=("decisao_correta", "sum"))
             .assign(taxa=lambda d: (100 * d.acertos / d.n).round(1)))

print(percepcao.to_string())
print(f"\ngeral: {corridas['decisao_correta'].sum()}/{len(corridas)} "
      f"({100*corridas['decisao_correta'].mean():.1f}%)")

                         n  acertos   taxa
classe_pasta                              
R-25a_esquerda          10       10  100.0
R-25b_direita           10       10  100.0
R-26_frente             10        4   40.0
R-3_sentido-proibido    10       10  100.0
R-4a_proibido-esquerda  10        6   60.0
R-4b_proibido-direita   10       10  100.0

geral: 50/60 (83.3%)


### 1.3 — Decomposição dos erros

Nem todo erro é confusão do modelo. Há dois modos de falha distintos:
classificar a placa errada, ou nunca chegar a decidir — quando a corrida
estoura o tempo limite sem que a bounding box atinja o limiar de disparo.
As colunas `classe_detectada` e `duracao_corrida_s` separam os dois casos.

In [4]:
erros = corridas[~corridas["decisao_correta"]]
print(erros[["run", "classe_pasta", "classe_detectada", "comando_emitido",
             "confianca_decisao", "duracao_corrida_s", "n_frames"]]
      .to_string(index=False))

  run           classe_pasta      classe_detectada comando_emitido  confianca_decisao  duracao_corrida_s  n_frames
run01            R-26_frente                  None            None                NaN              10.08        50
run02            R-26_frente                  None            None                NaN              10.45        48
run04            R-26_frente                  None            None                NaN              10.15        49
run06            R-26_frente                  None            None                NaN              10.13        46
run08            R-26_frente                  None            None                NaN              10.07        48
run09            R-26_frente                  None            None                NaN              10.78        25
run02 R-4a_proibido-esquerda                  None            None                NaN              10.05        39
run03 R-4a_proibido-esquerda                  None            None              

### 1.4 — O que o modelo via nas corridas que estouraram o tempo

Para as corridas sem decisão, a pergunta é se o modelo não via a placa ou
se via e o gatilho nunca disparou. `com deteccao` conta os quadros com
detecção; `bbox_w max` mostra a maior largura de caixa alcançada, que é o
quanto a corrida chegou perto do limiar.

In [5]:
runs_timeout = erros[erros["classe_detectada"].isna()][["run", "classe_pasta"]]

for _, linha in runs_timeout.iterrows():
    f = frames[(frames["run"] == linha["run"]) &
               (frames["classe_pasta"] == linha["classe_pasta"])]
    com_det = f[f["classe_detectada"].notna()]
    print(f"{linha['classe_pasta']:24s} {linha['run']}  "
          f"frames={len(f):3d}  com deteccao={len(com_det):3d}  "
          f"classes={com_det['classe_detectada'].unique().tolist()}  "
          f"bbox_w max={com_det['bbox_w'].max() if len(com_det) else '-'}")

R-26_frente              run01  frames= 50  com deteccao= 16  classes=['R-26_frente', 'R-25b_direita']  bbox_w max=66.0
R-26_frente              run02  frames= 48  com deteccao= 14  classes=['R-26_frente']  bbox_w max=86.0
R-26_frente              run04  frames= 49  com deteccao= 10  classes=['R-26_frente']  bbox_w max=65.0
R-26_frente              run06  frames= 46  com deteccao= 26  classes=['R-26_frente']  bbox_w max=73.0
R-26_frente              run08  frames= 48  com deteccao= 11  classes=['R-26_frente', 'R-25b_direita']  bbox_w max=68.0
R-26_frente              run09  frames= 25  com deteccao= 10  classes=['R-26_frente', 'R-4a_proibido-esquerda']  bbox_w max=78.0
R-4a_proibido-esquerda   run02  frames= 39  com deteccao= 18  classes=['R-4a_proibido-esquerda', 'R-3_sentido-proibido']  bbox_w max=296.0
R-4a_proibido-esquerda   run03  frames= 53  com deteccao= 53  classes=['R-4a_proibido-esquerda']  bbox_w max=70.0
R-4a_proibido-esquerda   run05  frames= 36  com deteccao= 36  classes

### 1.5 — Latência por etapa

`t_captura_ms` é o intervalo entre iterações consecutivas do laço principal,
ou seja, o período do laço inteiro — ele **contém** as demais etapas, que o
decompõem em vez de se somarem a ele. `t_comando_ms` tem *n* menor porque só
é cronometrado nos quadros em que houve envio.

In [6]:
etapas = ["t_captura_ms", "t_inferencia_ms", "t_decisao_ms", "t_comando_ms"]

lat = frames[etapas].describe().T[["count", "mean", "std", "min", "max"]]
lat["p95"] = frames[etapas].quantile(0.95)
print(lat[["count", "mean", "std", "p95", "min", "max"]].round(2).to_string())

                  count    mean     std     p95     min      max
t_captura_ms     1260.0  227.72  101.50  353.94  114.37  2144.57
t_inferencia_ms  1260.0  163.95   66.14  241.13  101.96  2004.11
t_decisao_ms     1260.0    0.02    0.41    0.01    0.00    14.70
t_comando_ms      715.0   94.79   76.80  161.79    2.30  1009.78


## Seção 2 — Comparação de latência entre modelos

Os quatro pipelines rodam sobre o mesmo vídeo, na mesma máquina, um quadro
por vez — que é o regime real de operação, já que o robô não processa lotes.
Cada modelo é aquecido antes da medição: a primeira inferência inclui
alocação de memória na GPU e construção do grafo, e distorceria a média.

Detecção em uma etapa (YOLO) contra duas etapas (Hough propõe a região, a
rede classifica). O tempo medido é o do pipeline como implementado — a
transformada de Hough roda em CPU e as redes em GPU.

In [7]:
import time
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from ultralytics import YOLO

VIDEO = "data/video_robocar.mp4"
TAM = 80
MAX_FRAMES = 2000     # amostra: latencia por frame nao precisa do video inteiro

linhas = []

# ===================== YOLOv8n ===========================================

m = YOLO("models/yolov8n.pt")
m.predict(np.zeros((720, 1280, 3), np.uint8), verbose=False)   # aquecimento

cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter("analise/video_yolov8n.mp4",
                      cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
tempos = []
while len(tempos) < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()                              # cronometro abre
    vis = m.predict(frame, conf=0.5, verbose=False)[0].plot()
    tempos.append((time.perf_counter() - t0) * 1000)      # e fecha aqui
    out.write(vis)                                        # gravacao fora
    # \r volta o cursor ao inicio da linha: o contador se reescreve no lugar
    print(f"\rYOLOv8n        frame {len(tempos):4d}/{MAX_FRAMES}", end="", flush=True)
cap.release()
out.release()

t = np.array(tempos)
print()
print(f"YOLOv8n      media={t.mean():7.2f}  dp={t.std():6.2f}  p95={np.percentile(t,95):7.2f}  n={len(t)}")
linhas.append({"modelo": "YOLOv8n", "media_ms": round(t.mean(), 2),
               "dp_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2)})

# ===================== YOLO11n ===========================================

m = YOLO("models/yolov11n.pt")
m.predict(np.zeros((720, 1280, 3), np.uint8), verbose=False)

cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter("analise/video_yolo11n.mp4",
                      cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
tempos = []
while len(tempos) < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()
    vis = m.predict(frame, conf=0.5, verbose=False)[0].plot()
    tempos.append((time.perf_counter() - t0) * 1000)
    out.write(vis)
    print(f"\rYOLO11n        frame {len(tempos):4d}/{MAX_FRAMES}", end="", flush=True)
cap.release()
out.release()

t = np.array(tempos)
print()
print(f"YOLO11n      media={t.mean():7.2f}  dp={t.std():6.2f}  p95={np.percentile(t,95):7.2f}  n={len(t)}")
linhas.append({"modelo": "YOLO11n", "media_ms": round(t.mean(), 2),
               "dp_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2)})

# ===================== Hough + MLP =======================================

m = tf.keras.models.load_model("models/mlp.keras")
m(np.zeros((1, TAM, TAM, 1), np.float32), training=False)

cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter("analise/video_hough_mlp.mp4",
                      cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
tempos = []
while len(tempos) < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()
    vis = frame.copy()
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    suave = cv2.medianBlur(cinza, 3)
    c = cv2.HoughCircles(suave, cv2.HOUGH_GRADIENT_ALT, 1, 60,
                         param1=120, param2=0.9, minRadius=10, maxRadius=80)
    if c is not None:
        cx, cy, r = [int(v) for v in np.round(c[0][0])]
        lado = int(r * 1.25)
        rec = suave[max(cy-lado, 0):cy+lado, max(cx-lado, 0):cx+lado]
        if rec.size:
            redim = cv2.resize(rec, (TAM, TAM))
            _, binaria = cv2.threshold(redim, 0, 255, cv2.THRESH_OTSU)
            entrada = (binaria.astype(np.float32) / 255).reshape(1, TAM, TAM, 1)
            m(entrada, training=False)     # classifica; rotulo nao e usado
            cv2.rectangle(vis, (cx-lado, cy-lado), (cx+lado, cy+lado), (0, 255, 0), 2)
    tempos.append((time.perf_counter() - t0) * 1000)
    out.write(vis)
    print(f"\rHough + MLP    frame {len(tempos):4d}/{MAX_FRAMES}", end="", flush=True)
cap.release()
out.release()

t = np.array(tempos)
print()
print(f"Hough + MLP  media={t.mean():7.2f}  dp={t.std():6.2f}  p95={np.percentile(t,95):7.2f}  n={len(t)}")
linhas.append({"modelo": "Hough + MLP", "media_ms": round(t.mean(), 2),
               "dp_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2)})

# ===================== Hough + CNN =======================================

m = tf.keras.models.load_model("models/cnn.keras")
m(np.zeros((1, TAM, TAM, 1), np.float32), training=False)

cap = cv2.VideoCapture(VIDEO)
out = cv2.VideoWriter("analise/video_hough_cnn.mp4",
                      cv2.VideoWriter_fourcc(*"mp4v"), cap.get(cv2.CAP_PROP_FPS),
                      (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                       int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))))
tempos = []
while len(tempos) < MAX_FRAMES:
    ok, frame = cap.read()
    if not ok:
        break
    t0 = time.perf_counter()
    vis = frame.copy()
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    suave = cv2.medianBlur(cinza, 3)
    c = cv2.HoughCircles(suave, cv2.HOUGH_GRADIENT_ALT, 1, 60,
                         param1=120, param2=0.9, minRadius=10, maxRadius=80)
    if c is not None:
        cx, cy, r = [int(v) for v in np.round(c[0][0])]
        lado = int(r * 1.25)
        rec = suave[max(cy-lado, 0):cy+lado, max(cx-lado, 0):cx+lado]
        if rec.size:
            redim = cv2.resize(rec, (TAM, TAM))
            _, binaria = cv2.threshold(redim, 0, 255, cv2.THRESH_OTSU)
            entrada = (binaria.astype(np.float32) / 255).reshape(1, TAM, TAM, 1)
            m(entrada, training=False)
            cv2.rectangle(vis, (cx-lado, cy-lado), (cx+lado, cy+lado), (0, 255, 0), 2)
    tempos.append((time.perf_counter() - t0) * 1000)
    out.write(vis)
    print(f"\rHough + CNN    frame {len(tempos):4d}/{MAX_FRAMES}", end="", flush=True)
cap.release()
out.release()

t = np.array(tempos)
print()
print(f"Hough + CNN  media={t.mean():7.2f}  dp={t.std():6.2f}  p95={np.percentile(t,95):7.2f}  n={len(t)}")
linhas.append({"modelo": "Hough + CNN", "media_ms": round(t.mean(), 2),
               "dp_ms": round(t.std(), 2), "p95_ms": round(np.percentile(t, 95), 2)})

# ===================== TABELA FINAL ======================================

print()
print(pd.DataFrame(linhas).to_string(index=False))

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8n        frame 2000/2000
YOLOv8n      media=  12.41  dp=  2.86  p95=  18.73  n=2000
YOLO11n        frame 2000/2000
YOLO11n      media=  15.35  dp=  5.26  p95=  22.91  n=2000
Hough + MLP    frame 2000/2000
Hough + MLP  media=  69.55  dp= 19.92  p95= 104.59  n=2000
Hough + CNN    frame 2000/2000
Hough + CNN  media=  68.03  dp= 18.09  p95= 105.24  n=2000

     modelo  media_ms  dp_ms  p95_ms
    YOLOv8n     12.41   2.86   18.73
    YOLO11n     15.35   5.26   22.91
Hough + MLP     69.55  19.92  104.59
Hough + CNN     68.03  18.09  105.24


### 2.1 — Tamanho dos modelos

O tamanho do arquivo de pesos complementa a latência: os dois eixos juntos
descrevem o custo de embarcar cada abordagem.

In [8]:
for nome, caminho in [("YOLOv8n", "models/yolov8n.pt"),
                      ("YOLO11n", "models/yolov11n.pt"),
                      ("MLP",     "models/mlp.keras"),
                      ("CNN",     "models/cnn.keras")]:
    p = Path(caminho)
    if p.exists():
        print(f"{nome:10s} {p.stat().st_size/1e6:7.1f} MB")

YOLOv8n        6.3 MB
YOLO11n        5.5 MB
MLP           52.0 MB
CNN           78.8 MB
